# 09 · Agent Ops & LangSmith — 动手实验

> 完整目录见 [`README.md`](README.md) · 建议在 **02**、**06** 之后学习


> 📖 完整讲义已嵌入下方章节（原 `agent-ops-langsmith.md` 已删除）
本 Notebook 模拟 LangSmith 的核心功能，无需真实 API Key。

**学习目标**
1. 理解 Trace / Run 数据模型
2. 自定义 Metrics 与告警
3. Human-in-the-Loop (HITL) 流程
4. 成本感知与优雅降级
5. Regression 检测与 Dataset 评估

In [ ]:
# ============================================================
# Part 0  Setup — mock LangSmith client + shared helpers
# ============================================================
import uuid, time, random, json, threading
from datetime import datetime, timezone
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List

# ---------- Mock LangSmith Data Model ----------
@dataclass
class Run:
    id: str
    name: str
    run_type: str          # 'chain' | 'llm' | 'tool' | 'retriever'
    inputs: Dict[str, Any]
    outputs: Dict[str, Any] = field(default_factory=dict)
    error: Optional[str] = None
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    extra: Dict[str, Any] = field(default_factory=dict)
    parent_run_id: Optional[str] = None
    session_name: str = 'default'

    @property
    def latency_ms(self):
        if self.end_time:
            return (self.end_time - self.start_time) * 1000
        return None

    @property
    def tokens(self):
        return self.extra.get('tokens', 0)

    @property
    def cost_usd(self):
        return self.extra.get('cost_usd', 0.0)


class MockLangSmithClient:
    """In-memory LangSmith client that stores runs & feedback."""

    def __init__(self):
        self._runs: Dict[str, Run] = {}
        self._feedback: List[Dict] = []
        self._datasets: Dict[str, List] = {}
        self._lock = threading.Lock()

    # --- Run API ---
    def create_run(self, name, run_type, inputs, parent_run_id=None,
                   session_name='default', extra=None) -> Run:
        run = Run(
            id=str(uuid.uuid4())[:8],
            name=name,
            run_type=run_type,
            inputs=inputs,
            parent_run_id=parent_run_id,
            session_name=session_name,
            extra=extra or {},
        )
        with self._lock:
            self._runs[run.id] = run
        return run

    def update_run(self, run_id, outputs=None, error=None, extra_patch=None):
        with self._lock:
            run = self._runs[run_id]
            run.end_time = time.time()
            if outputs:
                run.outputs = outputs
            if error:
                run.error = error
            if extra_patch:
                run.extra.update(extra_patch)

    def get_run(self, run_id) -> Run:
        return self._runs[run_id]

    def list_runs(self, session_name=None, run_type=None):
        with self._lock:
            runs = list(self._runs.values())
        if session_name:
            runs = [r for r in runs if r.session_name == session_name]
        if run_type:
            runs = [r for r in runs if r.run_type == run_type]
        return runs

    # --- Feedback API ---
    def create_feedback(self, run_id, key, score, comment=''):
        fb = dict(run_id=run_id, key=key, score=score,
                  comment=comment, created_at=datetime.now(timezone.utc).isoformat())
        with self._lock:
            self._feedback.append(fb)
        return fb

    def get_feedback(self, run_id=None):
        with self._lock:
            fb = list(self._feedback)
        if run_id:
            fb = [f for f in fb if f['run_id'] == run_id]
        return fb

    # --- Dataset API ---
    def create_dataset(self, name):
        self._datasets[name] = []
        return name

    def create_example(self, dataset_name, inputs, outputs):
        ex = dict(id=str(uuid.uuid4())[:8], inputs=inputs, outputs=outputs)
        self._datasets[dataset_name].append(ex)
        return ex

    def list_examples(self, dataset_name):
        return self._datasets.get(dataset_name, [])


# Global client for this notebook
ls = MockLangSmithClient()
print('MockLangSmithClient ready')


## 📖 讲义

# Agent Operations（Agent Ops）
# 与 LangSmith

## 多 Agent 系统的监控、调试与持续优化

<br>

**AI Engineer 训练营 · 进阶模块**
面向零基础学员 · **1.5 小时**

---

## Part 1  Tracing — 记录 Run / Trace 树

LangSmith 的核心是 **Run**，每次 LLM 调用、工具调用、Chain 执行都产生一个 Run。
多个 Run 通过 `parent_run_id` 链接成一棵 **Trace 树**。

In [ ]:
# ============================================================
# Part 1  基础 Tracing
# ============================================================
import random, time

PRICING = {
    'gpt-4o':      {'input': 2.50/1e6, 'output': 10.0/1e6},
    'gpt-4o-mini': {'input': 0.15/1e6, 'output':  0.60/1e6},
    'claude-3-haiku': {'input': 0.25/1e6, 'output': 1.25/1e6},
}

def fake_llm(prompt, model='gpt-4o-mini', latency_range=(0.2, 0.8)):
    """Simulate an LLM call and return (text, token_counts, cost)."""
    time.sleep(random.uniform(*latency_range))
    in_tok  = len(prompt.split())
    out_tok = random.randint(30, 120)
    price   = PRICING[model]
    cost    = in_tok * price['input'] + out_tok * price['output']
    return f'[{model}] Answer for: {prompt[:40]}...', in_tok + out_tok, cost

def rag_pipeline(question: str, session='demo') -> dict:
    """3-step RAG: retrieve -> rerank -> generate, each step is a Run."""
    # Parent chain run
    chain_run = ls.create_run('rag_chain', 'chain',
                              inputs={'question': question},
                              session_name=session)

    # Step 1 — retriever
    ret_run = ls.create_run('retriever', 'retriever',
                            inputs={'query': question},
                            parent_run_id=chain_run.id,
                            session_name=session)
    time.sleep(0.05)
    docs = [f'doc_{i}' for i in range(5)]
    ls.update_run(ret_run.id, outputs={'docs': docs})

    # Step 2 — reranker (tool)
    rerank_run = ls.create_run('reranker', 'tool',
                               inputs={'docs': docs, 'query': question},
                               parent_run_id=chain_run.id,
                               session_name=session)
    time.sleep(0.03)
    top_docs = docs[:3]
    ls.update_run(rerank_run.id, outputs={'top_docs': top_docs})

    # Step 3 — LLM generation
    prompt = f'Context: {top_docs}\nQuestion: {question}'
    llm_run = ls.create_run('generator_llm', 'llm',
                            inputs={'prompt': prompt},
                            parent_run_id=chain_run.id,
                            session_name=session)
    answer, tokens, cost = fake_llm(prompt)
    ls.update_run(llm_run.id,
                  outputs={'text': answer},
                  extra_patch={'tokens': tokens, 'cost_usd': cost})

    # Close chain run
    ls.update_run(chain_run.id,
                  outputs={'answer': answer},
                  extra_patch={'tokens': tokens, 'cost_usd': cost})

    return {'run_id': chain_run.id, 'answer': answer,
            'tokens': tokens, 'cost_usd': cost}


# Run 5 queries
questions = [
    'What is vector search?',
    'How does RAG work?',
    'Explain attention mechanism',
    'What is a knowledge graph?',
    'Compare BM25 and dense retrieval',
]
results = [rag_pipeline(q, session='part1') for q in questions]

# Print trace summary
print('--- Trace Summary ---')
for r in results:
    run = ls.get_run(r['run_id'])
    print(f'[{run.id}] latency={run.latency_ms:.0f}ms '
          f'tokens={run.tokens} cost=${run.cost_usd:.5f}')

all_runs = ls.list_runs(session_name='part1')
print(f'\nTotal runs stored: {len(all_runs)}')
print('Run types:', {rt: sum(1 for r in all_runs if r.run_type == rt)
                    for rt in ['chain','retriever','tool','llm']})


## 📖 讲义

# 课程地图

| 时间 | Part | 内容 |
|------|------|------|
| 0:00–0:10 | 引言 | Agent Ops 是什么？为什么必须有？ |
| 0:10–0:25 | Part 1 | Agent Ops 的六大职责 |
| 0:25–0:45 | Part 2 | LangSmith 核心功能详解 |
| 0:45–1:00 | Part 3 | Tracing & Debugging 实战 |
| 1:00–1:15 | Part 4 | 监控、指标与告警 |
| 1:15–1:25 | Part 5 | 质量门、HITL 与持续优化 |
| 1:25–1:35 | Part 6 | 成本控制 & 调度策略 |
| 1:35–1:45 | Part 7 | 练习导引 & 课程小结 |

> 配套 Notebook 模拟 LangSmith 全功能，**无需 API Key 即可运行**

---

---

# 引言
## Agent Ops 是什么？为什么必须有？

---

# 从"能跑"到"可运维"

**开发阶段**（你已经学会的）：
```
用户 ──▶ Agent ──▶ 回答
```

**生产阶段**（真正的挑战）：
```
问题 1：今天上午 10 点那次查询为什么回答错了？
问题 2：Retriever 的 p95 延迟从 300ms 涨到 1200ms，是什么原因？
问题 3：用户投诉 Agent 胡说，怎么复现？
问题 4：新版模型上线后，质量有没有退化？
问题 5：昨天 API 费用突然涨了 3 倍，哪里调用异常了？
问题 6：合规审计要求查某次决策的完整链路，能查到吗？
```

**没有 Agent Ops，这些问题都无法回答。**

---

# Agent Ops 的定义

> **Agent Ops** = 把"能运行的 Agent"变成"可持续运营的生产系统"的一整套工程实践

类比：
- **DevOps** = 开发 + 运维
- **MLOps** = 机器学习 + 运维
- **Agent Ops** = AI Agent + 运维（含 LLM 特有挑战）

**LLM 特有的运维挑战**

| 传统系统 | LLM Agent 系统 |
|----------|----------------|
| 输出确定性强 | 输出概率性，难以单元测试 |
| 性能指标清晰 | 质量指标主观（faithfulness, relevancy） |
| 成本固定 | token 成本随用量变化，难预测 |
| 调试有栈追踪 | 决策链在 LLM 内部，需 trace 还原 |

---

## Part 2  自定义 Metrics 与告警

**六大指标维度**: 延迟 · Token 用量 · 成本 · 错误率 · 质量分 · 吞吐量

我们实现一个 `MetricsCollector`，支持窗口统计和阈值告警。

In [ ]:
# ============================================================
# Part 2  Metrics Collector + 告警
# ============================================================
from collections import deque

@dataclass
class Alert:
    level: str    # 'WARN' | 'CRITICAL'
    metric: str
    value: float
    threshold: float
    message: str

class MetricsCollector:
    def __init__(self, window=20):
        self.window = window
        self._latencies  = deque(maxlen=window)
        self._costs      = deque(maxlen=window)
        self._tokens     = deque(maxlen=window)
        self._errors     = deque(maxlen=window)
        self.alerts: List[Alert] = []

        # Thresholds
        self.thresholds = {
            'p95_latency_ms': {'warn': 800, 'critical': 1500},
            'error_rate':     {'warn': 0.05, 'critical': 0.15},
            'cost_per_call':  {'warn': 0.001, 'critical': 0.005},
        }

    def record(self, run: 'Run'):
        self._latencies.append(run.latency_ms or 0)
        self._costs.append(run.cost_usd)
        self._tokens.append(run.tokens)
        self._errors.append(1 if run.error else 0)
        self._check_alerts()

    def _percentile(self, data, p):
        if not data:
            return 0
        s = sorted(data)
        idx = int(len(s) * p / 100)
        return s[min(idx, len(s)-1)]

    def _check_alerts(self):
        # p95 latency
        p95 = self._percentile(list(self._latencies), 95)
        thr = self.thresholds['p95_latency_ms']
        if p95 > thr['critical']:
            self.alerts.append(Alert('CRITICAL', 'p95_latency_ms', p95, thr['critical'],
                                     f'p95 latency {p95:.0f}ms > {thr["critical"]}ms'))
        elif p95 > thr['warn']:
            self.alerts.append(Alert('WARN', 'p95_latency_ms', p95, thr['warn'],
                                     f'p95 latency {p95:.0f}ms > {thr["warn"]}ms'))

        # error rate
        if len(self._errors) >= 5:
            er = sum(self._errors) / len(self._errors)
            thr = self.thresholds['error_rate']
            if er > thr['critical']:
                self.alerts.append(Alert('CRITICAL', 'error_rate', er, thr['critical'],
                                         f'Error rate {er:.1%} > {thr["critical"]:.1%}'))
            elif er > thr['warn']:
                self.alerts.append(Alert('WARN', 'error_rate', er, thr['warn'],
                                         f'Error rate {er:.1%} > {thr["warn"]:.1%}'))

    def summary(self):
        lat = list(self._latencies)
        return {
            'p50_latency_ms': self._percentile(lat, 50),
            'p95_latency_ms': self._percentile(lat, 95),
            'avg_cost_usd':   sum(self._costs) / len(self._costs) if self._costs else 0,
            'total_tokens':   sum(self._tokens),
            'error_rate':     sum(self._errors) / len(self._errors) if self._errors else 0,
            'sample_count':   len(lat),
        }


mc = MetricsCollector(window=20)

# Simulate 30 calls with occasional errors and latency spikes
def simulate_calls(n=30, error_prob=0.1, spike_prob=0.15):
    for i in range(n):
        is_error = random.random() < error_prob
        is_spike = random.random() < spike_prob
        latency  = random.uniform(1200, 2000) if is_spike else random.uniform(200, 700)

        run = ls.create_run(f'call_{i}', 'chain',
                            inputs={'q': f'question_{i}'}, session_name='metrics_sim')
        time.sleep(latency / 1000)
        if is_error:
            ls.update_run(run.id, error='TimeoutError')
        else:
            cost = random.uniform(0.0001, 0.003)
            tok  = random.randint(200, 800)
            ls.update_run(run.id, outputs={'ok': True},
                          extra_patch={'cost_usd': cost, 'tokens': tok})
        mc.record(ls.get_run(run.id))

simulate_calls(30)

print('=== Metrics Summary ===')
for k, v in mc.summary().items():
    print(f'  {k:25s}: {v:.4f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print(f'\n=== Alerts ({len(mc.alerts)}) ===')
shown = set()
for a in mc.alerts:
    key = (a.level, a.metric)
    if key not in shown:
        shown.add(key)
        print(f'  [{a.level}] {a.message}')


## 📖 讲义

# Part 1
## Agent Ops 的六大职责

---

# 六大职责全景

<div class="three-col">
<div>

### 1. 观察 Observability
收集 runs / traces
记录 prompt, retrieval, output
保存工具调用参数与返回

### 2. 监控 Monitoring
延迟 p50/p95/p99
错误率 & 工具调用频率
Token 用量 & cost
RAGAS 质量指标

</div>
<div>

### 3. 调试 Debugging
trace 回放，定位失败步骤
查看 Thought/Action/Observation
下载 artifacts 本地复现

### 4. 调度 Scheduling
实时队列 vs 批处理队列
优先级与资源分配
自动扩缩容策略

</div>
<div>

### 5. 治理 Governance
权限与访问控制
合规审计日志
PII 脱敏策略
HITL 人机协作

### 6. 持续优化 Continuous Ops
自动化评估
回归检测（CI/CD 集成）
告警与变更控制
A/B 实验

</div>
</div>

---

# 职责 1 & 2：观察与监控

**关键指标体系**

| 维度 | 指标 | 告警阈值示例 |
|------|------|-------------|
| **延迟** | p50/p95/p99 latency per agent | p95 > 5s → WARN |
| **错误** | error_rate, tool_failure_rate | error_rate > 1% → ALERT |
| **质量** | verifier_pass_rate, ragas_score | pass_rate < 0.8 → REVIEW |
| **成本** | tokens_per_call, cost_per_job | daily_cost > budget × 0.8 → WARN |
| **吞吐** | requests/s, queue_length | queue > 500 → scale_out |

**黄金信号（Four Golden Signals）**
```
Latency   ← 请求耗时（关注尾部，p99）
Traffic   ← 请求量（requests/s, jobs/hour）
Errors    ← 错误率（%）
Saturation← 资源饱和度（CPU, 队列长度, token quota）
```

---

# 职责 3：调试——还原 Agent 决策链

**没有 trace 时的调试**：

```
用户说：这个答案是错的
你说：为什么？
Agent 说：我不知道
你说：...
```

**有 LangSmith trace 时**：

```
job_id: j-001  agent: orchestrator  latency: 3.2s

  step 1 │ retriever     │ input: "差旅政策"
         │               │ output: [doc1, doc2, doc3]  ✅

  step 2 │ summarizer    │ input: [doc1, doc2, doc3]
         │               │ model: claude-haiku
         │               │ tokens: 1250 in, 320 out
         │               │ output: "差旅需提前申请..."  ✅

  step 3 │ verifier      │ claim: "出差免审批"
         │               │ nli_result: NOT ENTAILED   ❌ ← 问题在这里
         │               │ pass_rate: 0.33
```

---

# 职责 4 & 5：调度与治理

**调度设计**：按任务类型分队列

```
realtime_queue  ─▶  [Worker × 4]   ← SLA < 2s，用户直接等待
batch_queue     ─▶  [Worker × 2]   ← SLA < 10min，后台离线
priority_queue  ─▶  [Worker × 1]   ← VIP 用户专用
```

**治理关键点**

```
合规要求                           技术实现
─────────────────────────────────────────────────
保留 7 年审计日志           →   Append-only DB + 加密
PII 不得进入 LLM            →   Retriever 输出处脱敏
所有 AI 决策可追溯           →   LangSmith trace + job_id
高风险决策需人工确认          →   HITL + 审核结果写回 trace
```

---

---

# Part 2
## LangSmith 核心功能详解

---

# LangSmith 是什么？

> LangSmith = LangChain 官方的 **LLM 应用可观测性平台**
> 提供：run tracing、artifact 存储、evaluation、dataset 管理、告警

**核心概念**

| 概念 | 说明 |
|------|------|
| **Project** | 工作空间（如 `prod-rag-system`），聚合所有 runs |
| **Run** | 一次 Agent/Chain/LLM 调用的完整记录 |
| **Trace** | 一个 Run 及其所有子 Run 的树状结构 |
| **Artifact** | 附加到 Run 的文件（prompt、retrieved docs、output） |
| **Dataset** | 用于评估的输入/输出对集合 |
| **Feedback** | 对 Run 的评分（手动或自动） |

---

# LangSmith 接入方式

**方式 1：环境变量（最简单，推荐开发）**

```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY="ls__xxx"
export LANGCHAIN_PROJECT="my-agent-project"
```

```python
# 仅需设置环境变量，所有 LangChain 调用自动上报
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(model="gpt-4o-mini")
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
result = chain.invoke({"query": "什么是差旅政策？"})
# ↑ 自动上报到 LangSmith，无需额外代码
```

**方式 2：显式 tracer（精细控制）**

```python
from langsmith import traceable

@traceable(name="retriever_agent", tags=["prod", "v2.1"])
def retriever_agent(query: str) -> dict:
    hits = vector_store.search(query, k=5)
    return {"hits": hits, "query": query}
```

---

# LangSmith Run 记录什么？

**一个完整的 Run 包含**：

```json
{
  "run_id":     "uuid",
  "name":       "orchestrator",
  "run_type":   "chain",
  "start_time": "2025-01-15T10:30:00Z",
  "end_time":   "2025-01-15T10:30:03.2Z",
  "inputs":  { "query": "生成差旅政策报告" },
  "outputs": { "report": "...", "pass_rate": 0.917 },
  "error":   null,
  "extra": {
    "metadata": {
      "job_id":      "j-001",
      "model":       "claude-sonnet-4-6",
      "env":         "prod",
      "agent_version": "2.1.0"
    }
  },
  "child_runs": [
    { "name": "retriever",   "latency_ms": 210 },
    { "name": "summarizer",  "latency_ms": 302 },
    { "name": "verifier",    "latency_ms": 145 }
  ]
}
```

---

# 手动记录自定义 Metric & Artifact

有时需要记录 LangChain 自动 trace 之外的信息：

```python
from langsmith import Client
from langsmith.run_helpers import get_current_run_tree

client = Client()

@traceable(name="verifier_agent")
def verify_report(report: dict, job_id: str) -> dict:
    claims    = extract_claims(report)
    results   = [nli_check(c) for c in claims]
    pass_rate = sum(results) / len(results)

    # 记录自定义 metric
    run = get_current_run_tree()
    if run:
        run.add_metadata({
            "pass_rate":      pass_rate,
            "claims_checked": len(claims),
            "job_id":         job_id,
        })
        # 记录 artifact：详细的 NLI 结果
        client.create_feedback(
            run.id,
            key="verifier_pass_rate",
            score=pass_rate,
            comment=f"{len(claims)} claims checked",
        )

    return {"pass": pass_rate >= 0.8, "pass_rate": pass_rate}
```

---

# Dataset & Evaluation（离线评估）

**创建 Dataset**：

```python
from langsmith import Client

client = Client()

# 创建评估数据集
dataset = client.create_dataset("rag-quality-v1")

# 添加测试样例
client.create_examples(
    inputs=[
        {"query": "差旅政策的审批时限是多少？"},
        {"query": "采购金额超过多少需要 CFO 审批？"},
    ],
    outputs=[
        {"expected": "国内出差 3 天，国际出差 10 天"},
        {"expected": "超过 5 万元需经部门经理和 CFO 双签"},
    ],
    dataset_id=dataset.id,
)
```

**对比 baseline vs new_model**：

```python
from langsmith.evaluation import evaluate

results = evaluate(
    target=my_agent_chain,
    data="rag-quality-v1",
    evaluators=["exact_match", "embedding_similarity"],
    experiment_prefix="v2.1-vs-v2.0",
)
```

---

## Part 3  Human-in-the-Loop (HITL)

当质量分低于阈值时，自动创建人工审核票据（Ticket），
审核员通过 `Feedback API` 回写分数，系统决定放行或拦截。

In [ ]:
# ============================================================
# Part 3  HITL — 质量门 + 人工审核
# ============================================================
import queue, threading

@dataclass
class ReviewTicket:
    ticket_id: str
    run_id: str
    question: str
    answer: str
    auto_score: float    # 0-1, automated quality score
    status: str = 'pending'  # pending | approved | rejected
    reviewer: str = ''
    review_comment: str = ''

class HITLGateway:
    QUALITY_THRESHOLD = 0.6   # below this => human review required

    def __init__(self, client: MockLangSmithClient):
        self.client = client
        self.tickets: List[ReviewTicket] = []
        self._queue: queue.Queue = queue.Queue()
        self._approved_outputs: Dict[str, str] = {}

    def evaluate_output(self, run_id, question, answer) -> float:
        """Simulate automated quality scoring (e.g. faithfulness check)."""
        # Heuristic: longer answers = higher score (demo only)
        base = min(1.0, len(answer.split()) / 50)
        return round(base + random.uniform(-0.15, 0.15), 2)

    def submit(self, run_id, question, answer) -> ReviewTicket:
        score = self.evaluate_output(run_id, question, answer)
        ticket = ReviewTicket(
            ticket_id=str(uuid.uuid4())[:6],
            run_id=run_id, question=question, answer=answer,
            auto_score=score,
            status='approved' if score >= self.QUALITY_THRESHOLD else 'pending',
        )
        self.tickets.append(ticket)
        if ticket.status == 'pending':
            self._queue.put(ticket)
        # Record feedback
        self.client.create_feedback(
            run_id=run_id, key='quality_score', score=score,
            comment=f'auto: {"pass" if score >= self.QUALITY_THRESHOLD else "review"}',
        )
        return ticket

    def review(self, ticket_id, approved: bool, reviewer='alice', comment='') -> ReviewTicket:
        """Human reviewer approves or rejects a ticket."""
        ticket = next(t for t in self.tickets if t.ticket_id == ticket_id)
        ticket.status = 'approved' if approved else 'rejected'
        ticket.reviewer = reviewer
        ticket.review_comment = comment
        self.client.create_feedback(
            run_id=ticket.run_id, key='human_approved', score=1 if approved else 0,
            comment=f'{reviewer}: {comment}',
        )
        return ticket

    def stats(self):
        total    = len(self.tickets)
        auto_ok  = sum(1 for t in self.tickets if t.status == 'approved' and not t.reviewer)
        reviewed = sum(1 for t in self.tickets if t.reviewer)
        rejected = sum(1 for t in self.tickets if t.status == 'rejected')
        pending  = sum(1 for t in self.tickets if t.status == 'pending')
        return dict(total=total, auto_approved=auto_ok, human_reviewed=reviewed,
                    rejected=rejected, pending=pending)


hitl = HITLGateway(ls)

# Simulate 10 RAG outputs going through the quality gate
test_cases = [
    ('What is RLHF?', 'RLHF stands for Reinforcement Learning from Human Feedback. It fine-tunes LLMs using human preference signals to improve alignment and helpfulness.'),
    ('Explain embeddings', 'Embeddings.'),          # short -> low score -> review
    ('What is LangChain?', 'LangChain is a framework for building applications powered by large language models, supporting chains, agents, and memory components.'),
    ('How does FAISS work?', 'FAISS (Facebook AI Similarity Search) is an efficient library for similarity search and clustering of dense vectors, optimized for speed and memory.'),
    ('What is fine-tuning?', 'Fine.'),              # very short -> low
    ('Describe vector DB', 'A vector database stores high-dimensional embeddings and supports approximate nearest-neighbor search, used in semantic search and RAG pipelines.'),
    ('What is prompt engineering?', 'It is the art of writing prompts.'),  # borderline
    ('Explain MoE', 'Mixture of Experts (MoE) is a model architecture where different expert networks handle different parts of the input, improving capacity without linear cost scaling.'),
    ('What is LangSmith?', 'LangSmith is an observability and evaluation platform by LangChain for tracing, debugging, and monitoring LLM applications in production.'),
    ('Describe RAG pipeline', 'RAG combines dense retrieval with generative models: retrieve relevant documents, rerank them, then generate a grounded answer conditioned on the retrieved context.'),
]

for q, a in test_cases:
    run = ls.create_run('qa_chain', 'chain', inputs={'q': q}, session_name='hitl_demo')
    ls.update_run(run.id, outputs={'answer': a})
    ticket = hitl.submit(run.id, q, a)
    status = ticket.status.upper()
    print(f'[{ticket.ticket_id}] score={ticket.auto_score:.2f} {status:8s} | {q[:40]}')

# Human reviews all pending tickets
pending = [t for t in hitl.tickets if t.status == 'pending']
print(f'\n-- {len(pending)} tickets need human review --')
for t in pending:
    # Simulate reviewer deciding based on answer length
    approve = len(t.answer.split()) > 3
    hitl.review(t.ticket_id, approved=approve, reviewer='alice',
                comment='sufficient detail' if approve else 'answer too short')
    print(f'  [{t.ticket_id}] human => {"APPROVED" if approve else "REJECTED"}')

print('\n=== HITL Stats ===')
for k, v in hitl.stats().items():
    print(f'  {k}: {v}')


## 📖 讲义

# Part 3
## Tracing & Debugging 实战

---

# Trace 结构：树 vs 链

**树状 Trace（LangSmith 实际结构）**：

```
run: orchestrator  [3.2s]
├── run: retriever         [0.21s]  ✅
├── run: summarizer        [0.30s]  ✅
│     └── run: llm_call    [0.28s]  model=haiku  tokens=1250→320
├── run: risk_extractor    [0.25s]  ✅
│     └── run: llm_call    [0.23s]  model=haiku  tokens=890→180
└── run: verifier          [0.14s]  ⚠️ pass_rate=0.58
      ├── tool: nli_check  claim_1  entailed=True
      ├── tool: nli_check  claim_2  entailed=False  ← 失败
      └── tool: nli_check  claim_3  entailed=False  ← 失败
```

**调试流程**：
1. 在 LangSmith UI 找到该 run（按 job_id 过滤）
2. 展开 verifier → 看 claim_2/claim_3 的 input
3. 查看 NLI context：是检索到了错误文档，还是断言本身有误？
4. 下载 artifact 在本地重现

---

# 调试技巧：关键字段查看

**在 LangSmith UI 调试时重点看**：

| 检查项 | 路径 | 排查什么 |
|--------|------|----------|
| LLM 输入 prompt | run → inputs → messages | prompt 是否正确？context 是否注入？ |
| Retrieved 文档 | run.metadata.retrieved_docs | 检索到的是相关内容吗？ |
| LLM 输出 | run → outputs → content | 模型回答了什么？是否 hallucination？ |
| Tool 调用参数 | child_run inputs | 工具被调用时参数是否合法？ |
| Tool 返回 | child_run outputs | 工具返回了什么？是否报错？ |
| Latency 分布 | run metrics | 哪个 step 最慢？ |

**常见问题定位**

```
延迟突然变高   → 看 llm_call latency：是模型响应慢，还是检索慢？
回答质量变差   → 看 retrieved_docs：检索到的文档是否相关？
工具调用失败   → 看 tool_call inputs：参数格式是否正确？
verifier 失败  → 看 nli_check context：支撑文本是否充分？
```

---

# 回放（Replay）：本地复现问题

**从 LangSmith 下载 artifacts 本地复现**：

```python
from langsmith import Client

client = Client()

# 找到失败的 run
failed_run = client.read_run("run-id-xxx")

# 提取原始输入
original_query    = failed_run.inputs["query"]
retrieved_context = failed_run.metadata.get("retrieved_docs", [])
llm_prompt        = failed_run.inputs.get("messages", [])

# 本地重跑（方便加断点调试）
def replay_locally(run_id: str):
    run   = client.read_run(run_id)
    query = run.inputs["query"]

    # 直接用相同 retrieved_context，排除检索变量
    answer = generator.run(
        question=query,
        context=run.metadata["retrieved_docs"],
    )
    print(f"Original: {run.outputs['answer']}")
    print(f"Replay:   {answer}")
    print(f"Same?     {answer == run.outputs['answer']}")
```

---

---

# Part 4
## 监控、指标与告警

---

# 监控体系：从采集到告警

```
Agent 运行
   │
   ▼
LangSmith Trace  ──▶  Run DB（存储所有 runs + metrics）
   │                        │
   │                 定时聚合 Job（每 5min）
   │                        │
   ▼                        ▼
Prometheus Metrics    告警规则引擎
   │                        │
   ▼                        ▼
Grafana Dashboard    Slack / PagerDuty / Email
```

**采样策略**（降低成本）：

| 场景 | 采样率 | 说明 |
|------|--------|------|
| 正常请求 | 10% full trace | 压缩存储成本 |
| 高延迟请求（> p95） | 100% | 异常必须全记 |
| 错误请求 | 100% | 失败必须全记 |
| HITL 触发 | 100% | 合规需要 |
| 新模型上线期 | 100% | 观察期全量 |

---

# RAGAS 质量指标体系

**四个核心 RAGAS 指标**

| 指标 | 含义 | 计算方式 |
|------|------|----------|
| **Faithfulness** | 答案是否有文档支撑 | LLM 判断每个句子是否来自 context |
| **Answer Relevancy** | 答案是否切题 | 答案与问题的语义相似度 |
| **Context Precision** | 检索到的 context 是否都有用 | 有用 chunk / 总 chunk 数 |
| **Context Recall** | 相关内容是否都检索到了 | 召回率（需 ground truth） |

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

result = evaluate(
    dataset=test_dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=eval_llm,
    embeddings=embeddings,
)
# result: {'faithfulness': 0.87, 'answer_relevancy': 0.91}
```

---

# 告警策略：分级响应

```python
ALERT_RULES = [
    {
        "name":      "latency_p95_high",
        "metric":    "p95_latency_ms",
        "threshold": 5000,
        "window":    "5m",
        "action":    "auto_scale_out",   # 自动扩容
        "severity":  "WARN",
    },
    {
        "name":      "error_rate_spike",
        "metric":    "error_rate_pct",
        "threshold": 3.0,
        "window":    "5m",
        "action":    "page_oncall",      # 立即呼叫值班
        "severity":  "CRITICAL",
    },
    {
        "name":      "ragas_regression",
        "metric":    "mean_faithfulness",
        "threshold": "baseline * 0.95",  # 下降 5%
        "window":    "1h",
        "action":    "block_deploy",     # 阻断新版本部署
        "severity":  "HIGH",
    },
    {
        "name":      "daily_cost_80pct",
        "metric":    "daily_cost_usd",
        "threshold": "budget * 0.80",
        "window":    "24h",
        "action":    "notify_team",      # 通知团队
        "severity":  "WARN",
    },
]
```

---

# 回归检测：CI/CD 集成

**每次模型更新/Prompt 变更都跑回归**：

```python
# ci_evaluate.py（在 GitHub Actions 中运行）
from langsmith.evaluation import evaluate
from langsmith import Client

client = Client()

def run_regression(new_chain, baseline_scores: dict):
    result = evaluate(
        target=new_chain,
        data="rag-quality-v1",       # 固定测试集
        evaluators=["faithfulness", "answer_relevancy"],
    )

    passed = True
    for metric, baseline in baseline_scores.items():
        current = result.scores[metric]
        if current < baseline * 0.97:   # 允许 3% 波动
            print(f"REGRESSION: {metric} {baseline:.3f} -> {current:.3f}")
            passed = False

    if not passed:
        raise SystemExit(1)   # CI 失败，阻断部署

# 在 PR 中运行
run_regression(
    new_chain=create_chain(model="claude-sonnet-4-6"),
    baseline_scores={"faithfulness": 0.87, "answer_relevancy": 0.91}
)
```

---

## Part 4  成本感知 + 优雅降级

实现三层 **成本控制** 策略:
1. **预算追踪** — 实时累计 Token/Cost，接近上限时触发告警  
2. **模型降级** — 超预算时自动切换到更便宜的模型  
3. **速率限制** — 令牌桶算法控制每秒最大调用次数

In [ ]:
# ============================================================
# Part 4  成本感知调度 + 优雅降级
# ============================================================

# ---- Budget Tracker ----
class BudgetTracker:
    def __init__(self, daily_limit_usd=1.0, warn_pct=0.8):
        self.daily_limit  = daily_limit_usd
        self.warn_pct     = warn_pct
        self.spent        = 0.0
        self.total_tokens = 0
        self.call_count   = 0

    @property
    def remaining(self): return max(0, self.daily_limit - self.spent)

    @property
    def utilization(self): return self.spent / self.daily_limit

    def record(self, cost_usd, tokens):
        self.spent        += cost_usd
        self.total_tokens += tokens
        self.call_count   += 1

    def tier(self):
        u = self.utilization
        if u >= 1.0:   return 'EXHAUSTED'
        if u >= 0.9:   return 'CRITICAL'
        if u >= self.warn_pct: return 'WARN'
        return 'OK'


# ---- Token Bucket Rate Limiter ----
class TokenBucket:
    def __init__(self, rate=5, capacity=10):
        self.rate     = rate      # tokens per second
        self.capacity = capacity
        self.tokens   = capacity
        self.last_ts  = time.time()
        self._lock    = threading.Lock()

    def consume(self, n=1) -> bool:
        with self._lock:
            now = time.time()
            refill = (now - self.last_ts) * self.rate
            self.tokens = min(self.capacity, self.tokens + refill)
            self.last_ts = now
            if self.tokens >= n:
                self.tokens -= n
                return True
            return False


# ---- Cost-Aware LLM Router ----
MODEL_CASCADE = ['gpt-4o', 'gpt-4o-mini', 'claude-3-haiku']

class CostAwareRouter:
    def __init__(self, budget: BudgetTracker, rate_limiter: TokenBucket):
        self.budget = budget
        self.rl     = rate_limiter
        self.degradation_log: List[dict] = []

    def select_model(self, preferred='gpt-4o') -> str:
        tier = self.budget.tier()
        if tier == 'OK':
            return preferred
        elif tier == 'WARN':
            # Downgrade one level
            idx = MODEL_CASCADE.index(preferred)
            return MODEL_CASCADE[min(idx + 1, len(MODEL_CASCADE) - 1)]
        elif tier in ('CRITICAL', 'EXHAUSTED'):
            return MODEL_CASCADE[-1]  # cheapest
        return preferred

    def call(self, prompt, preferred='gpt-4o', session='cost_demo'):
        if not self.rl.consume():
            return None, 'RATE_LIMITED'
        model = self.select_model(preferred)
        if model != preferred:
            self.degradation_log.append({
                'from': preferred, 'to': model,
                'budget_tier': self.budget.tier(),
                'utilization': self.budget.utilization,
            })
        run = ls.create_run('llm_call', 'llm',
                            inputs={'prompt': prompt, 'model': model},
                            session_name=session)
        answer, tokens, cost = fake_llm(prompt, model=model, latency_range=(0.05, 0.2))
        ls.update_run(run.id, outputs={'text': answer},
                      extra_patch={'tokens': tokens, 'cost_usd': cost, 'model': model})
        self.budget.record(cost, tokens)
        return run.id, model


budget = BudgetTracker(daily_limit_usd=0.005, warn_pct=0.6)  # small limit for demo
rl     = TokenBucket(rate=100, capacity=100)  # high rate so we don't block
router = CostAwareRouter(budget, rl)

print('Simulating 20 calls with gpt-4o preferred...')
print(f'{"Call":>5}  {"Budget%":>8}  {"Tier":>10}  {"Model Selected":>20}  Degraded?')
print('-' * 65)
for i in range(20):
    run_id, model_used = router.call(f'Query {i}', preferred='gpt-4o')
    degraded = model_used != 'gpt-4o' if run_id else False
    print(f'{i+1:>5}  {budget.utilization:>7.1%}  {budget.tier():>10}  {str(model_used):>20}  {"YES" if degraded else ""}')

print(f'\nTotal degradations: {len(router.degradation_log)}')
print(f'Final spend: ${budget.spent:.5f} / ${budget.daily_limit}')
print(f'Total tokens: {budget.total_tokens}')


## 📖 讲义

# Part 5
## 质量门、HITL 与持续优化

---

# 质量门：自动 vs 人工

```
Run 完成
   │
   ▼
Quality Gate 自动评估
   │
   ├─ faithfulness >= 0.85  ✅
   ├─ answer_relevancy >= 0.80  ✅
   ├─ latency_ms <= 5000  ✅
   └─ pass_rate >= 0.80  ⚠️ 0.67 < 0.80
         │
         ▼
   HITL 触发（REVIEW 级）
         │
         ├─ 在 LangSmith 中标记该 run 为 "needs_review"
         ├─ 发送 Slack 通知：run_id + 失败原因 + 快速链接
         └─ 挂起输出，等待人工审核
               │
               ▼
         人工审核结果写回 LangSmith Feedback
         client.create_feedback(run_id, key="human_review",
                                score=1.0, comment="approved")
```

---

# HITL 与 LangSmith 集成

```python
from langsmith import Client

client = Client()

def handle_low_quality_run(run_id: str, pass_rate: float):
    # 1. 标记 run 需要审核
    client.create_feedback(
        run_id=run_id,
        key="quality_flag",
        score=pass_rate,
        comment=f"pass_rate={pass_rate:.2f} below threshold 0.80",
    )

    # 2. 发送通知（Slack webhook）
    send_slack(
        channel="#ai-ops",
        text=f":warning: Low quality run {run_id[:8]}\n"
             f"pass_rate={pass_rate:.2f}\n"
             f"Review: https://smith.langchain.com/runs/{run_id}"
    )

    # 3. 审核人在 LangSmith UI 操作后，回调记录决策
    # (webhook 或轮询)

def record_human_decision(run_id: str, decision: str, reviewer: str):
    client.create_feedback(
        run_id=run_id,
        key="human_review_decision",
        value=decision,          # "approve" / "reject" / "revise"
        comment=f"Reviewed by {reviewer}",
    )
```

---

---

# Part 6
## 成本控制 & 调度策略

---

# Token 成本管理

**三层成本控制**

```
Layer 1: 模型路由（按任务选最便宜够用的模型）
  ─────────────────────────────────────────────
  summarize_short  → claude-haiku-4-5   ← 3x cheaper
  risk_extract     → claude-sonnet-4-6  ← 主力
  complex_reason   → claude-opus-4-6    ← 按需

Layer 2: 语义缓存（相似问题直接返回缓存）
  ─────────────────────────────────────────────
  cache_key = embedding_hash(query, threshold=0.95)
  if cache.exists(key): return cache.get(key)  # 0 token

Layer 3: 预算门（超出预算降级）
  ─────────────────────────────────────────────
  if daily_spend > budget * 0.8:
      switch_to_haiku()    # 降级到小模型
  if daily_spend > budget:
      return cached_or_error()  # 完全降级
```

---

# 调度策略：队列分层

```
                ┌─────────────────┐
  用户请求 ──▶  │  Router          │
                └───┬─────┬───────┘
                    │     │
              ┌─────▼─┐ ┌─▼──────┐
              │Realtime│ │ Batch  │
              │Queue   │ │Queue   │
              │SLA<2s  │ │SLA<10m │
              └────┬───┘ └───┬────┘
                   │         │
            ┌──────▼──┐ ┌───▼──────┐
            │Worker×4 │ │Worker×2  │
            │(高性能)  │ │(低成本)  │
            └─────────┘ └──────────┘
```

**K8s HPA 基于队列长度自动扩缩**：

```yaml
metrics:
  - type: External
    external:
      metric:
        name: sqs_approximate_number_of_messages_visible
      target:
        type: AverageValue
        averageValue: "50"   # 每个 Pod 处理 50 条消息
```

---

# 运维 Checklist

**每次发布前**：

```
□ 跑离线评估（RAGAS）并与 baseline 对比
□ 确认 trace 采样率配置正确
□ 确认告警规则已更新（新模型的阈值）
□ 确认成本预算已更新
□ 确认 HITL webhook 可用

每天例行：
□ 查看昨日 p95 latency / error rate 趋势
□ 查看 daily token cost
□ 扫描 HITL pending tickets
□ 检查 ragas_score 是否有漂移

每周例行：
□ 查看 top 10 失败 runs（按 error_type 分组）
□ 审查 human_review 结果，更新 quality gate 阈值
□ 评估采样率是否合适（成本 vs 覆盖率）
□ Review audit log 合规性
```

---

## Part 5  Regression 检测 + Dataset 评估

**Regression 检测**：对比当前版本与 baseline 版本在标准 Dataset 上的指标。

评估指标（简化 RAGAS 风格）：
- **Faithfulness** — 答案是否来自上下文
- **Answer Relevancy** — 答案是否回答了问题
- **Conciseness** — 答案是否简洁

In [ ]:
# ============================================================
# Part 5  Dataset 评估 & Regression 检测
# ============================================================

# Build evaluation dataset
ds_name = ls.create_dataset('qa_eval_v1')
eval_cases = [
    ('What is RAG?',
     'RAG (Retrieval-Augmented Generation) retrieves relevant documents and conditions generation on them.'),
    ('What is LangSmith?',
     'LangSmith is an observability platform for LLM applications providing tracing, evaluation, and monitoring.'),
    ('Explain embeddings',
     'Embeddings are dense vector representations of text that capture semantic meaning in a continuous space.'),
    ('What is a vector database?',
     'A vector database indexes embeddings for fast approximate nearest-neighbor search, enabling semantic retrieval.'),
    ('What is prompt engineering?',
     'Prompt engineering designs input prompts to guide LLM behavior and improve output quality.'),
]
for q, ref in eval_cases:
    ls.create_example(ds_name, inputs={'question': q}, outputs={'reference': ref})


# ---- Evaluators ----
def score_faithfulness(answer, context_keywords):
    """Proxy: fraction of context keywords present in answer."""
    if not context_keywords: return 0.5
    hits = sum(1 for kw in context_keywords if kw.lower() in answer.lower())
    return hits / len(context_keywords)

def score_relevancy(question, answer):
    """Proxy: fraction of question words present in answer."""
    q_words = set(question.lower().split()) - {'what','is','a','the','how','does','an'}
    if not q_words: return 0.5
    hits = sum(1 for w in q_words if w in answer.lower())
    return min(1.0, hits / len(q_words))

def score_conciseness(answer, max_words=60):
    return min(1.0, max_words / max(len(answer.split()), 1))


def run_evaluation(model='gpt-4o-mini', noise=0.0, version='v1'):
    """Run the eval dataset and return per-example + aggregate metrics."""
    examples = ls.list_examples(ds_name)
    results = []
    for ex in examples:
        q   = ex['inputs']['question']
        ref = ex['outputs']['reference']
        # Simulate answer (add noise to simulate model quality variation)
        answer, tokens, cost = fake_llm(q, model=model, latency_range=(0.01, 0.05))
        # Use reference keywords as proxy context
        context_kw = ref.split()[:8]
        # Add noise to scores to simulate regression
        faith = min(1.0, max(0.0, score_faithfulness(ref, context_kw) + random.gauss(0, noise)))
        relev = min(1.0, max(0.0, score_relevancy(q, ref) + random.gauss(0, noise)))
        conci = score_conciseness(answer)
        results.append({'question': q, 'faithfulness': faith,
                        'relevancy': relev, 'conciseness': conci,
                        'tokens': tokens, 'cost': cost})
    agg = {
        'version': version,
        'model': model,
        'faithfulness': sum(r['faithfulness'] for r in results) / len(results),
        'relevancy':    sum(r['relevancy']    for r in results) / len(results),
        'conciseness':  sum(r['conciseness']  for r in results) / len(results),
        'avg_tokens':   sum(r['tokens'] for r in results) / len(results),
        'total_cost':   sum(r['cost']   for r in results),
    }
    return agg, results


def detect_regression(baseline, current, threshold=0.05):
    """Flag any metric that drops by more than threshold vs baseline."""
    regressions = []
    for metric in ['faithfulness', 'relevancy', 'conciseness']:
        delta = current[metric] - baseline[metric]
        if delta < -threshold:
            regressions.append({
                'metric': metric,
                'baseline': baseline[metric],
                'current':  current[metric],
                'delta':    delta,
            })
    return regressions


# Run baseline (v1, low noise)
baseline_agg, _ = run_evaluation(model='gpt-4o-mini', noise=0.02, version='v1_baseline')
# Simulate a degraded version (v2, higher noise + worse model)
current_agg, _  = run_evaluation(model='gpt-4o-mini', noise=0.15, version='v2_candidate')

print('=== Evaluation Results ===')
print(f'{"Metric":20s}  {"Baseline (v1)":>15}  {"Current (v2)":>14}  {"Delta":>8}')
print('-' * 65)
for m in ['faithfulness', 'relevancy', 'conciseness']:
    delta = current_agg[m] - baseline_agg[m]
    flag  = '<-- REGRESSION' if delta < -0.05 else ''
    print(f'{m:20s}  {baseline_agg[m]:>15.3f}  {current_agg[m]:>14.3f}  {delta:>+8.3f}  {flag}')
print(f'{"avg_tokens":20s}  {baseline_agg["avg_tokens"]:>15.0f}  {current_agg["avg_tokens"]:>14.0f}')
print(f'{"total_cost":20s}  ${baseline_agg["total_cost"]:>14.5f}  ${current_agg["total_cost"]:>13.5f}')

regressions = detect_regression(baseline_agg, current_agg)
if regressions:
    print('\n[ALERT] Regression detected! Block v2 deployment.')
    for r in regressions:
        print(f'  {r["metric"]}: {r["baseline"]:.3f} -> {r["current"]:.3f} (delta={r["delta"]:+.3f})')
else:
    print('\n[OK] No regression detected. v2 is safe to deploy.')


## 📖 讲义

# Part 7
## 练习导引 & 课程小结

---

# 四个课堂练习

| # | 练习 | 时长 | 目标 |
|---|------|------|------|
| **1** | LangSmith 接入 + trace 查看 | 30–60 min | 会在 UI 中查找失败步骤 |
| **2** | 自定义 metric + Slack 告警 | 60–90 min | 能检测 pass_rate 回归并告警 |
| **3** | HITL 流程：low quality → ticket → 审核 | 90–120 min | 完整 HITL 链路，结果写回 LangSmith |
| **4** | 成本感知：超预算自动降级 | 60–90 min | 累计 cost > 80% 预算时切换小模型 |

> 配套 Notebook 已内置**完整模拟版 LangSmith**，无需真实 API Key 即可完成所有练习

---

# 课程小结

**Agent Ops 的核心价值**

```
开发阶段：能运行 ✅
生产阶段：可观测 + 可调试 + 可告警 + 可审计 + 持续优化 ✅
```

**LangSmith 在 Agent Ops 中的角色**

| 功能 | LangSmith 提供 |
|------|----------------|
| Trace 存储 | Run tree + artifacts（prompt, context, output） |
| 调试 | UI 可视化链路，支持 filter/replay |
| 评估 | Dataset + evaluators + 历史对比 |
| HITL | Feedback API + 审核结果记录 |
| 告警 | 基于 metrics 的阈值告警（需配合脚本） |

**三条铁律**
1. **没有 trace 就没有调试** — 每个 Agent 每步必记
2. **没有回归检测就没有安全发布** — 新模型上线必跑 eval
3. **没有 HITL 就没有高风险治理** — 低质量自动转人工

---

# 推荐资源

<div class="columns">
<div>

### LangSmith
- 官方文档：`docs.smith.langchain.com`
- Python SDK：`pip install langsmith`
- 免费 tier：1000 traces/month

### 评估框架
- **RAGAS** — RAG 评估专用
- **TruLens** — LLM 应用通用评估
- **DeepEval** — 单元测试风格评估

### 替代工具
- **Langfuse** — 开源可自建（GDPR 友好）
- **Phoenix (Arize)** — 开源 LLM observability
- **Helicone** — OpenAI 专用代理监控

</div>
<div>

### Metrics & Alerting
- **Prometheus + Grafana** — 标准 metrics 栈
- **Datadog** — 商业全栈监控
- **Sentry** — 错误追踪

### 成本管理
- **LangSmith** 内置 token 统计
- **OpenMeter** — 用量计费
- 各 LLM 提供商的 usage dashboard

### 工作流编排
- **Prefect / Temporal** — 生产工作流
- **Airflow** — 批处理调度
- **Celery + Redis** — 任务队列

</div>
</div>

---

---

# Q & A

## 有什么问题？

<br>

**课后作业**
- 练习 1：接入 LangSmith（或 Notebook 模拟版），查看 3 次 run 的 trace
- 练习 2：实现 pass_rate 回归检测 + Slack 告警（可用 print 模拟）

<br>

*下一课：RAG 评估框架与持续质量优化*

## Part 6  综合演练 — Production Agent Ops 流水线

将前面所有模块串联：
```
Request -> RateLimiter -> CostRouter -> RAGPipeline
        -> MetricsCollector -> QualityGate (HITL)
        -> LangSmith Feedback
```

In [ ]:
# ============================================================
# Part 6  综合 Agent Ops 流水线
# ============================================================

class AgentOpsPipeline:
    def __init__(self):
        self.budget  = BudgetTracker(daily_limit_usd=0.05, warn_pct=0.7)
        self.rl      = TokenBucket(rate=50, capacity=50)
        self.router  = CostAwareRouter(self.budget, self.rl)
        self.metrics = MetricsCollector(window=50)
        self.hitl    = HITLGateway(ls)
        self.request_log: List[dict] = []

    def handle(self, question: str) -> dict:
        # 1. Rate check
        if not self.rl.consume():
            return {'status': 'RATE_LIMITED', 'question': question}

        # 2. RAG pipeline with cost-aware model
        model = self.router.select_model('gpt-4o-mini')
        chain_run = ls.create_run('ops_chain', 'chain',
                                  inputs={'question': question, 'model': model},
                                  session_name='ops_pipeline')
        answer, tokens, cost = fake_llm(question, model=model, latency_range=(0.05, 0.3))
        ls.update_run(chain_run.id, outputs={'answer': answer},
                      extra_patch={'tokens': tokens, 'cost_usd': cost, 'model': model})
        self.budget.record(cost, tokens)

        # 3. Metrics
        run_obj = ls.get_run(chain_run.id)
        self.metrics.record(run_obj)

        # 4. Quality gate
        ticket = self.hitl.submit(chain_run.id, question, answer)

        entry = {
            'run_id': chain_run.id, 'model': model,
            'tokens': tokens, 'cost': cost,
            'ticket_status': ticket.status,
            'auto_score': ticket.auto_score,
        }
        self.request_log.append(entry)
        return entry

    def report(self):
        print('=== Agent Ops Pipeline Report ===')
        print(f'  Requests processed : {len(self.request_log)}')
        print(f'  Budget utilization : {self.budget.utilization:.1%}')
        print(f'  Total cost         : ${self.budget.spent:.5f}')
        print(f'  Degradations       : {len(self.router.degradation_log)}')
        ms = self.metrics.summary()
        print(f'  p50 latency        : {ms["p50_latency_ms"]:.0f}ms')
        print(f'  p95 latency        : {ms["p95_latency_ms"]:.0f}ms')
        hs = self.hitl.stats()
        print(f'  Auto-approved      : {hs["auto_approved"]}')
        print(f'  Pending review     : {hs["pending"]}')
        if self.metrics.alerts:
            print(f'  Active alerts      : {len(set((a.level,a.metric) for a in self.metrics.alerts))}')
            seen = set()
            for a in self.metrics.alerts:
                k = (a.level, a.metric)
                if k not in seen:
                    seen.add(k)
                    print(f'    [{a.level}] {a.message}')


pipeline = AgentOpsPipeline()

batch_questions = [
    'What is LangSmith?', 'Explain RAG', 'How does RLHF work?',
    'What is chain-of-thought prompting?', 'Describe model distillation',
    'What is semantic search?', 'How do agents use tools?',
    'Explain temperature in LLMs', 'What is context window?',
    'How does FAISS indexing work?', 'What is tool calling?',
    'Explain mixture of experts', 'What is a system prompt?',
    'How does beam search work?', 'What is tokenization?',
]

print('Processing batch...\n')
for i, q in enumerate(batch_questions):
    result = pipeline.handle(q)
    status = result.get('ticket_status', result.get('status', '?')).upper()
    score  = result.get('auto_score', 0)
    model  = result.get('model', '-')
    print(f'[{i+1:02d}] {status:10s} score={score:.2f} model={model:20s} | {q[:40]}')

print()
pipeline.report()


## 总结

| 模块 | 核心概念 | 实现方式 |
|------|----------|----------|
| Tracing | Run / Trace 树, parent_run_id | MockLangSmithClient |
| Metrics | p50/p95 延迟, 错误率, 成本 | MetricsCollector + deque |
| HITL | 质量门, Feedback API | HITLGateway + ReviewTicket |
| 成本控制 | 预算追踪, 模型降级, 令牌桶 | BudgetTracker + TokenBucket |
| Regression | Dataset 评估, 指标对比 | run_evaluation + detect_regression |
| 综合 | 全链路 Agent Ops 流水线 | AgentOpsPipeline |

**课后练习**
1. 修改 `QUALITY_THRESHOLD`，观察 HITL 触发频率变化
2. 调低 `daily_limit_usd`，观察模型降级时机
3. 在 `run_evaluation` 中引入更高 `noise`，触发 regression alert
4. 为 `AgentOpsPipeline` 添加 SLA 违规告警（p99 > 2000ms）